# 06 - Predictions for the kaggle submission

Last step: take the model we trained and saved in notebook 05, use it to predict `is_churn` for the actual test users, and save the result in the format kaggle expects.

The test set is `sample_submission_v2.csv` - it lists every `msno` we need to predict for, with `is_churn` as a placeholder (currently all `0`). We'll build the same features for these users as we did for the training set in notebook 04, run them through the model, and overwrite that placeholder column with our real predictions.

## Step 1: Load everything we need

- `sample_submission_v2.csv` - the users to predict for, and the exact format kaggle wants back
- `members_v3.csv` - demographics, same as notebook 04
- The two aggregated parquet files from notebooks 02 and 03
- The model we trained and saved in notebook 05

In [1]:
import pandas as pd
import joblib
from pathlib import Path

DATA = Path("../data/raw")
PROCESSED = Path("../data/processed")
MODELS = Path("../models")

submission = pd.read_csv(DATA / "sample_submission_v2.csv")
members = pd.read_csv(DATA / "members_v3.csv")
transactions_agg = pd.read_parquet(PROCESSED / "transactions_v2_agg.parquet")
user_logs_agg = pd.read_parquet(PROCESSED / "user_logs_v2_agg.parquet")

model = joblib.load(MODELS / "lgbm_baseline.pkl")

print(submission.shape)
submission.head()

(907471, 2)


,msno,is_churn
0,4n+fXlyJvfQnTeKXTWT507Ll4JVYGrOC8LHCfwBmPE4=,0
1,aNmbC1GvFUxQyQUidCVmfbQ0YeCuwkPzEdQ0RwWyeZM=,0
2,rFC9eSG/tMuzpre6cwcMLZHEYM89xY02qcz7HL4//jc=,0
3,WZ59dLyrQcE7ft06MZ5dj40BnlYQY7PHgg/54+HaCSE=,0
4,aky/Iv8hMp1/V/yQHLtaVuEmmAxkB5GuasQZePJ7NU4=,0


## Step 2: Build the same feature table as notebook 04

Same three merges as before, in the same order - the only difference is the base table is now `submission` instead of `train`. That gives us one row per test user, with demographics, payment history, and listening activity attached.

In [2]:
df = submission.merge(members, on='msno', how='left')
df = df.merge(transactions_agg, on='msno', how='left')
df = df.merge(user_logs_agg, on='msno', how='left')

print(df.shape)
df.head()

(907471, 23)


,msno,is_churn,city,bd,gender,registered_via,registration_init_time,num_transactions,total_actual_paid,total_plan_list_price,...,last_expire_date,num_days,num_25_sum,num_50_sum,num_75_sum,num_985_sum,num_100_sum,num_unq_sum,total_secs_sum,total_secs_mean
0,4n+fXlyJvfQnTeKXTWT507Ll4JVYGrOC8LHCfwBmPE4=,0,1.0,0.0,NaN,7.0,20150718.0,1.0,99.0,99.0,...,2017-04-18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,aNmbC1GvFUxQyQUidCVmfbQ0YeCuwkPzEdQ0RwWyeZM=,0,4.0,28.0,male,9.0,20051030.0,1.0,149.0,149.0,...,2017-04-30,31.0,550.0,176.0,125.0,131.0,1701.0,2273.0,500802.64,16154.923871
2,rFC9eSG/tMuzpre6cwcMLZHEYM89xY02qcz7HL4//jc=,0,4.0,34.0,male,7.0,20141101.0,1.0,99.0,99.0,...,2017-04-15,10.0,79.0,19.0,1.0,2.0,84.0,149.0,23814.27,2381.427000
3,WZ59dLyrQcE7ft06MZ5dj40BnlYQY7PHgg/54+HaCSE=,0,NaN,NaN,NaN,NaN,NaN,1.0,99.0,99.0,...,2017-04-27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,aky/Iv8hMp1/V/yQHLtaVuEmmAxkB5GuasQZePJ7NU4=,0,1.0,0.0,NaN,13.0,20161222.0,1.0,129.0,129.0,...,2017-04-21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Step 3: Clean up the same way as notebook 04

The model was trained on data cleaned a certain way, so the test data needs exactly the same treatment:

- Fill missing transaction/log columns with `0` (no row found = no activity)
- Clip `bd` (age) to a sensible 0-100 range, anything outside that becomes `NaN`
- Fill missing `gender` with `"unknown"`

In [3]:
fill_cols = [
    'num_transactions', 'total_actual_paid', 'total_plan_list_price',
    'is_cancel_sum', 'is_auto_renew_mean',
    'num_days', 'num_25_sum', 'num_50_sum', 'num_75_sum', 'num_985_sum',
    'num_100_sum', 'num_unq_sum', 'total_secs_sum', 'total_secs_mean',
]
df[fill_cols] = df[fill_cols].fillna(0)

df['bd'] = df['bd'].where(df['bd'].between(0, 100))
df['gender'] = df['gender'].fillna('unknown')

(df.isna().mean() * 100).round(2)

msno                       0.00
is_churn                   0.00
city                      12.38
bd                        12.43
gender                     0.00
registered_via            12.38
registration_init_time    12.38
num_transactions           0.00
total_actual_paid          0.00
total_plan_list_price      0.00
is_cancel_sum              0.00
is_auto_renew_mean         0.00
last_transaction_date      0.00
last_expire_date           0.00
num_days                   0.00
num_25_sum                 0.00
num_50_sum                 0.00
num_75_sum                 0.00
num_985_sum                0.00
num_100_sum                0.00
num_unq_sum                0.00
total_secs_sum             0.00
total_secs_mean            0.00
dtype: float64

## Step 4: Line up the columns with what the model expects

Same as notebook 05 - drop the id, label placeholder, and date columns, and mark `city`, `gender`, `registered_via` as `category` dtype. Doing the merges and drops in the same order as notebook 05 means the columns end up matching what the model was trained on.

In [4]:
drop_cols = ['msno', 'is_churn', 'last_transaction_date', 'last_expire_date']
cat_cols = ['city', 'gender', 'registered_via']

X_test = df.drop(columns=drop_cols)

for col in cat_cols:
    X_test[col] = X_test[col].astype('category')

X_test.dtypes

city                      category
bd                         float64
gender                    category
registered_via            category
registration_init_time     float64
num_transactions           float64
total_actual_paid          float64
total_plan_list_price      float64
is_cancel_sum              float64
is_auto_renew_mean         float64
num_days                   float64
num_25_sum                 float64
num_50_sum                 float64
num_75_sum                 float64
num_985_sum                float64
num_100_sum                float64
num_unq_sum                float64
total_secs_sum             float64
total_secs_mean            float64
dtype: object

## Step 5: Predict

Run the test users through the model and take the predicted probability of churn for each one - `predict_proba(...)[:, 1]`, same as the evaluation step in notebook 05.

In [5]:
preds = model.predict_proba(X_test)[:, 1]

print(preds[:5])
print("min/max:", preds.min(), preds.max())

[0.00025305 0.00577956 0.00037862 0.0002652  0.00458489]
min/max: 0.00010664255474185992 0.9994622533052463


## Step 6: Build the submission file

Kaggle wants a csv with exactly two columns - `msno` and `is_churn` - one row per test user, same as `sample_submission_v2.csv`. We just overwrite the placeholder `is_churn` column with our predictions and save it.

In [6]:
submission['is_churn'] = preds

submission.to_csv(PROCESSED / "submission.csv", index=False)
print("saved:", PROCESSED / "submission.csv")
submission.head()

saved: ../data/processed/submission.csv


,msno,is_churn
0,4n+fXlyJvfQnTeKXTWT507Ll4JVYGrOC8LHCfwBmPE4=,0.000253
1,aNmbC1GvFUxQyQUidCVmfbQ0YeCuwkPzEdQ0RwWyeZM=,0.005780
2,rFC9eSG/tMuzpre6cwcMLZHEYM89xY02qcz7HL4//jc=,0.000379
3,WZ59dLyrQcE7ft06MZ5dj40BnlYQY7PHgg/54+HaCSE=,0.000265
4,aky/Iv8hMp1/V/yQHLtaVuEmmAxkB5GuasQZePJ7NU4=,0.004585
